In [6]:
# LABORATÓRIO 03 — COMPLETANDO O ACO

# Objetivo: implementar partes importantes do ACO
# a partir do código estudado no Laboratório 01.

# 1. Representação da rede
import numpy as np
import random
import matplotlib.pyplot as plt

CUSTOS = np.array([
    [0, 2, 4, np.inf, np.inf, np.inf],
    [2, 0, 1, 5, np.inf, np.inf],
    [4, 1, 0, 2, 3, np.inf],
    [np.inf, 5, 2, 0, 1, 4],
    [np.inf, np.inf, 3, 1, 0, 2],
    [np.inf, np.inf, np.inf, 4, 2, 0]
])

ORIGEM = 0
DESTINO = 5

NUM_FORMIGAS = 20
NUM_ITERACOES = 50

ALPHA = 1.0
BETA = 2.0

TAXA_EVAPORACAO = 0.5
Q = 100

feromonio = np.ones_like(CUSTOS, dtype=float)
feromonio[CUSTOS == np.inf] = 0


# 2. Função de vizinhos

def obter_vizinhos(no):

    vizinhos = []

    for proximo in range(len(CUSTOS)):

        if proximo != no and CUSTOS[no][proximo] != np.inf:
            vizinhos.append(proximo)

    return vizinhos


# 3. Desafio 1 — Calcular a atratividade

def calcular_atratividade(no_atual, proximo):

    fer = feromonio[no_atual][proximo]
    custo = CUSTOS[no_atual][proximo]

    atratividade = fer**ALPHA * (1 / custo)**BETA

    return atratividade


# 4. Desafio 2 — Evaporação

def evaporar_feromonio():

    global feromonio

    # Reduz o feromônio de acordo com a taxa de evaporação.
    feromonio *= (1 - TAXA_EVAPORACAO)

    # Mantém os caminhos inexistentes com feromônio igual a zero.
    feromonio[CUSTOS == np.inf] = 0


# 5. Desafio 3 — Depósito

def depositar_feromonio(rota, custo):

    # Quanto menor o custo, maior será o depósito.
    deposito = Q / custo

    for i in range(len(rota) - 1):

        origem = rota[i]
        destino = rota[i + 1]

        # Adiciona o depósito de feromônio ao caminho utilizado.
        feromonio[origem][destino] += deposito
        feromonio[destino][origem] += deposito


# Função para calcular o custo total da rota

def calcular_custo(rota):

    custo_total = 0

    for i in range(len(rota) - 1):

        origem = rota[i]
        destino = rota[i + 1]

        custo_total += CUSTOS[origem][destino]

    return custo_total


# 6. Desafio 4 — Construir uma rota

def construir_rota():

    rota = [ORIGEM]
    atual = ORIGEM

    while atual != DESTINO:


        vizinhos = obter_vizinhos(atual)

        candidatos = [
            no for no in vizinhos
            if no not in rota
        ]

        if not candidatos:
            return None


        atratividades = [
            calcular_atratividade(atual, no)
            for no in candidatos
        ]


        soma_atratividades = sum(atratividades)

        probabilidades = [
            atratividade / soma_atratividades
            for atratividade in atratividades
        ]


        proximo = random.choices(
            candidatos,
            weights=probabilidades,
            k=1
        )[0]

        rota.append(proximo)
        atual = proximo

    return rota


# 7. Execução

melhor_rota = None
melhor_custo = float("inf")

for iteracao in range(NUM_ITERACOES):

    rotas = []

    for _ in range(NUM_FORMIGAS):

        rota = construir_rota()

        if rota is not None:

            custo = calcular_custo(rota)

            rotas.append((rota, custo))

            if custo < melhor_custo:

                melhor_custo = custo
                melhor_rota = rota.copy()

    evaporar_feromonio()

    for rota, custo in rotas:

        depositar_feromonio(
            rota,
            custo
        )


print("Melhor rota:", melhor_rota)
print("Melhor custo:", melhor_custo)


# Questões para serem respondidas:

# 1 - Por que a fórmula da atratividade utiliza 1 / custo
# em vez de utilizar diretamente o custo?
# R: Porque quanto menor o custo, maior deve ser a atratividade.
# Usando 1/custo, caminhos mais baratos recebem maior atratividade.

# 2 - O que acontece com a atratividade quando uma rota
# recebe mais feromônio?
# R: A atratividade aumenta, pois o feromônio influencia
# positivamente a escolha do caminho.

# 3 - Por que a função construir_rota() precisa impedir
# que a formiga visite novamente um nó que já está na rota?
# R: Para evitar ciclos e visitas desnecessárias aos mesmos nós,
# permitindo que a formiga construa uma rota válida até o destino.


Melhor rota: [0, 1, 2, 3, 4, 5]
Melhor custo: 8.0
